In [1]:
from sklearn.linear_model import LogisticRegression
from src.data import load_data, split_by_year
from src.features import add_ratios
from src.config import RISK_TREND
from src.metrics import evaluate
from src.scorecard import WOEBinner

In [2]:
train, val, test = (add_ratios(d) for d in split_by_year(load_data()))

In [5]:
# 1. bins from train only, then check them
binner = WOEBinner(RISK_TREND).fit(train, train["default"])
binner.iv()

mve_tl    1.327459
tl_ta     1.047409
quick     0.836434
ni_ta     0.769016
re_ta     0.538928
log_ta    0.087330
dtype: float64

In [6]:
W_tr, W_va, W_te = (binner.transform(d) for d in (train, val, test))

In [7]:
lr = LogisticRegression(max_iter = 1000).fit(W_tr, train["default"])

dict(zip(W_tr.columns, lr.coef_[0]))

{'mve_tl': np.float64(-0.6870232946872344),
 'tl_ta': np.float64(-0.11920438006153927),
 'ni_ta': np.float64(-0.5952907017744155),
 'quick': np.float64(-0.34110061073572967),
 're_ta': np.float64(-0.19038684264082814),
 'log_ta': np.float64(0.0033624237433481537)}

In [ ]:
import pandas as pd
pd.DataFrame({
    "train": evaluate(train["default"], lr.predict_proba(W_tr)[:,1]),
    "val":evaluate(val["default"], lr.predict_proba(W_va)[:,1]),
    "test":evaluate(test["default"], lr.predict_proba(W_te)[:,1]),
})